# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Define common paths
policies_dir = '/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/sop_documents'
chroma_db_dir = '/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/chroma_db'
outputs_json_path = '/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/outputs.json'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q langchain pypdf chromadb sentence-transformers

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [8]:
!pip install -q langchain-community pypdf chromadb sentence-transformers langchain-text-splitters

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Load the SOP documents
loader = DirectoryLoader(policies_dir, glob="**/*.md", loader_cls=TextLoader)
docs = loader.load()

# Split documents into chunks for embedding
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, # A common chunk size
    chunk_overlap  = 50, # A common overlap to maintain context
    length_function = len,
    is_separator_regex = False,
)

texts = text_splitter.split_documents(docs)

print(f"Loaded {len(docs)} documents and split into {len(texts)} chunks.")

# 2. Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3. Generate embeddings (this will be used later to build the vector index)
# For now, let's validate one embedding

# Validate embeddings by embedding a sample string and checking vector length
sample_text = "This is a test sentence for embedding validation."
sample_embedding = embedding_model.embed_query(sample_text)

print(f"Sample embedding vector length: {len(sample_embedding)}")
if len(sample_embedding) == 384:
    print("Embedding model initialized successfully and produces 384-dimensional vectors.")
else:
    print("Warning: Embedding vector length is not 384. Please check the model configuration.")

Loaded 13 documents and split into 64 chunks.


/tmp/ipykernel_2927/3285302740.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sample embedding vector length: 384
Embedding model initialized successfully and produces 384-dimensional vectors.


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [9]:
from langchain_community.vectorstores import Chroma

# 1. Create a persistent Chroma vector index
vector_db = Chroma.from_documents(
    documents=texts, # Your chunked documents from 3.2.1
    embedding=embedding_model, # Your embedding model from 3.2.1
    persist_directory=chroma_db_dir
)

# 2. Configure similarity search and validate the index
print(f"ChromaDB index created and persisted at: {chroma_db_dir}")

# Validate with count
actual_doc_count = vector_db._collection.count()
expected_doc_count = len(texts)
print(f"Expected document count: {expected_doc_count}, Actual document count: {actual_doc_count}")
if actual_doc_count == expected_doc_count:
    print("Validation successful: Document count matches.")
else:
    print("Validation failed: Document count mismatch!")

# Validate with a test similarity search
test_query = "I need a refund"
similarity_results = vector_db.similarity_search(test_query, k=1)

print(f"\nTest Query: '{test_query}'")
print(f"Top 1 similarity search result (page_content): {similarity_results[0].page_content[:200]}...")

ChromaDB index created and persisted at: /content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/chroma_db
Expected document count: 64, Actual document count: 64
Validation successful: Document count matches.

Test Query: 'I need a refund'
Top 1 similarity search result (page_content): ## Agent Steps
1. Verify the order identifier and purchase date.
2. Confirm the item condition and refund reason.
3. Check for prior refunds on the same order to avoid double processing.
4. Submit the...


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

# Define a sample customer query
customer_query = "My order hasn't arrived yet. What should I do?"

# Perform similarity search to get the top relevant document (k=1)
retrieved_docs = vector_db.similarity_search(customer_query, k=1)

# Extract the content of the top retrieved document
retrieved_context = retrieved_docs[0].page_content

# Define the prompt template with a placeholder for context
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Answer strictly using the following policy document: {context}"),
    ("user", "{query}")
])

# Augment the LLM prompt with the retrieved context
naive_rag_output = prompt_template.format_messages(
    context=retrieved_context,
    query=customer_query
)

print(f"Customer Query: {customer_query}")
print("\n--- Retrieved Context ---")
print(retrieved_context[:500] + "...") # Print first 500 chars of context for brevity
print("\n--- Augmented LLM Prompt ---")
for message in naive_rag_output:
    print(f"{message.type}: {message.content}")

Customer Query: My order hasn't arrived yet. What should I do?

--- Retrieved Context ---
## "Delivered But Not Received"
1. Confirm the shipping address on the order.
2. Ask the customer to check with neighbors, household members, and safe
   drop-off spots, and to allow 24 hours, as carriers sometimes scan early.
3. If still missing, open a carrier investigation and offer a replacement or
   refund per the shipping delays and refund procedures....

--- Augmented LLM Prompt ---
system: Answer strictly using the following policy document: ## "Delivered But Not Received"
1. Confirm the shipping address on the order.
2. Ask the customer to check with neighbors, household members, and safe
   drop-off spots, and to allow 24 hours, as carriers sometimes scan early.
3. If still missing, open a carrier investigation and offer a replacement or
   refund per the shipping delays and refund procedures.
human: My order hasn't arrived yet. What should I do?


---
## Save Artifacts for Downstream Notebooks

In [11]:
import json
import os

# Ensure the directory for outputs.json exists
drive_path_base = os.path.dirname(outputs_json_path)
os.makedirs(drive_path_base, exist_ok=True)

# Load existing outputs if the file exists, otherwise initialize an empty dict
outputs = {}
if os.path.exists(outputs_json_path):
    with open(outputs_json_path, 'r') as f:
        outputs = json.load(f)

# Add naive_rag_output to the outputs dictionary
# We need to convert the message objects to dictionaries for JSON serialization
outputs['naive_rag_output'] = [
    {"type": msg.type, "content": msg.content} for msg in naive_rag_output
]

# Save the updated outputs dictionary to outputs.json
with open(outputs_json_path, 'w') as f:
    json.dump(outputs, f, indent=4)

print(f"'naive_rag_output' successfully saved to {outputs_json_path}")

'naive_rag_output' successfully saved to /content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/outputs.json


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**